In [1]:
# SPDX-License-Identifier: CC-BY-4.0
# Code for "Active Continual Learning with Metaplastic Binary Bayesian Neural Networks"
# Kellian Cottart, Théo Ballet, Djohan Bonnet, Damien Querlioz
# Portions of the code are adapted from the Pytorch project (BSD-3-Clause)
# Author: Kellian Cottart <kellian.cottart@gmail.com>
# Date: 2025-30-01

In [2]:

import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import os
import seaborn as sns
import re
import json
import pandas as pd
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.05"
results_folder = "results-main-pmnist-1000tasks-100neurons"
df = pd.DataFrame()
# iterate through all root folders in the results folder
for folder in os.listdir(results_folder):
    current_path = os.path.join(results_folder, folder)
    # extract the name from the first config
    config_path = os.path.join(current_path, "config0/config.json")
    with open(config_path, "r") as f:
        config = json.load(f)
    # n_iterations is the number of config folders
    n_iterations = len([f for f in os.listdir(current_path) if f.startswith("config") and os.path.isdir(os.path.join(current_path, f))])
    
    # Add the row of parameters to the dataframe   
    field = [key for key in config.keys() if "ewc" in key]
    field_si = [key for key in config.keys() if key=="si"]
    row = {
        "path": current_path,
        "opt": config["optimizer"],
        "n_tasks": config["n_tasks"],
        "n_epochs": config["epochs"],
        "n_train_samples": config["n_train_samples"] if "n_train_samples" in config else 1,
        "n_test_samples": config["n_test_samples"] if "n_test_samples" in config else 1,
        "n_iterations": n_iterations,
        "batch_size": config["train_batch_size"],
        "method": list(config["network_params"]["active_learning"].keys())[0] if "active_learning" in config["network_params"] else "none",
        "N": str(config["optimizer_params"]["N"]) if "N" in config["optimizer_params"] else "none",
        # key containing ewc but not strictly equal to ewc
        "ewc": config[field[0]] if len(field) > 0 else "none",
        "si": config[field_si[0]] if len(field_si) > 0 else "none",
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

# One color for each path
colors = sns.color_palette("viridis", len(df["method"].unique()))
markers = ["D", "o", "s", "h", "^", "x", "v", "p", "*", "X", "D", "o", "s", "h", "^", "x", "v", "p", "*", "X"]

In [3]:
data = []
for idx, row in df.iterrows():
    path = row["path"]
    n_tasks = row["n_tasks"]
    n_epochs = row["n_epochs"]
    n_iterations = row["n_iterations"]
    full_accuracies = []
    full_roc_aucs = []
    full_footprint = []
    for it in range(n_iterations):
        current_it_path = os.path.join(path, f"config{it}")
        accuracy_path = os.path.join(current_it_path, "accuracy")
        uncertainty_path = os.path.join(current_it_path, "uncertainty")
        accuracies = []
        roc_aucs = []
        for task in range(n_tasks):
            for epoch in range(n_epochs):
                suffix = f"task={task}-epoch={epoch}.npy"
                accuracies.append(jnp.load(os.path.join(accuracy_path, "split=0-"+suffix)))
                epistemic_u = jnp.load(os.path.join(uncertainty_path, f"roc-auc-epistemic-{suffix}"))
                aleatoric_u = jnp.load(os.path.join(uncertainty_path, f"roc-auc-aleatoric-{suffix}"))
                roc_aucs.append(epistemic_u if epistemic_u != 0.0 else aleatoric_u)
        full_footprint.append(jnp.load(os.path.join(current_it_path, "memory_occupation.npy")))
        full_accuracies.append(jnp.array(accuracies))
        full_roc_aucs.append(jnp.array(roc_aucs))
    full_accuracies = jnp.array(full_accuracies)*100
    accuracy_array_mean = jnp.mean(full_accuracies, -1)[:, -1].mean(0)
    accuracy_array_std = jnp.mean(full_accuracies, -1)[:, -1].std(0)
    full_roc_aucs = jnp.array(full_roc_aucs)
    full_footprint = jnp.array(full_footprint)
    data.append((full_accuracies, full_accuracies[:, -1, :], full_roc_aucs, full_accuracies[:, 0, :], full_footprint.mean(0)))
# Add new columns to df
df["accuracies"] = [d[0] for d in data]
df["last_accuracies"] = [d[1] for d in data]
df["roc_aucs"] = [d[2] for d in data]
df["first_accuracies"] = [d[3] for d in data]
df["footprint"] = [d[4] for d in data]


In [4]:
def make_key(row):
    key = row["opt"]
    if row["ewc"] != "none":
        key += "-ewc"
    if row["si"] != "none":
        key += "-si"
    if key == "adam":
        key = "STE"
    return key

def compute_bwt(accuracies):
    acc = accuracies / 100
    n_tasks = acc.shape[1]
    bwt_per_task = jnp.stack(
        [acc[:, -1, t] - acc[:, t, t] for t in range(n_tasks - 1)],
        axis=1
    )
    bwt_per_run = bwt_per_task.mean(axis=1)
    return f"{bwt_per_run.mean():.4f} \\pm {bwt_per_run.std():.4f}"


rows = {}

for _, row in df.iterrows():
    key = make_key(row)

    last_accuracies = row["last_accuracies"][:, -5:]
    acc_per_run = jnp.mean(last_accuracies, axis=-1)
    acc_mean = acc_per_run.mean()
    acc_std = acc_per_run.std()

    roc_aucs = row["roc_aucs"]
    auc_mean = roc_aucs[-1].mean()
    auc_std = roc_aucs[-1].std()

    last_accuracies = row["last_accuracies"].mean(0)[-1]/100
    max_accuracies = row["accuracies"].mean(0).max()/100
    memory_rigidity = 1 / (max_accuracies - last_accuracies + 1e-5)
    
    # compute backward transfer for 5 last tasks
    acc_matrix = row["accuracies"]  # shape (S, N, N)
    backward_transfer_all = compute_bwt(acc_matrix)
    rows[key] = {
        "Average 5 last tasks (%)": f"${acc_mean:.2f} \\pm {acc_std:.2f}$",
        "Fashion-MNIST OOD detection (ROC-AUC)": f"${auc_mean:.4f} \\pm {auc_std:.4f}$",
        "Maximum Memory Rigidity Resilience": f"${memory_rigidity:.4f}$",
        "Backward Transfer": f"${backward_transfer_all}$",
    }


combined_table = pd.DataFrame.from_dict(rows, orient="index")
combined_table.index = (
    combined_table.index
    .str.replace("bayesbinn", "Bayes BiNN")
    .str.replace("bimu", "BiMU")
    .str.replace("mesu", "MESU")
    .str.replace("adamw", "Adam")
    .str.replace("adam", "STE")
    .str.replace("sgd-ewc", "EWC Online")
    .str.replace("sgd-si", "Synaptic Intelligence")
    .str.replace("sgd", "SGD")
    .str.replace("synapticmetaplasticity", "Synaptic Metaplasticity")
)
combined_table

,Average 5 last tasks (%),Fashion-MNIST OOD detection (ROC-AUC),Maximum Memory Rigidity Resilience,Backward Transfer
Bayes BiNN,$41.12 \pm 1.62$,$0.5661 \pm 0.1154$,$2.0389$,$-0.3897 \pm 0.0021$
EWC Online,$81.78 \pm 0.82$,$0.6563 \pm 0.1102$,$6.6309$,$-0.7132 \pm 0.0015$
MESU,$91.69 \pm 0.58$,$0.9504 \pm 0.0253$,$261.1046$,$-0.8429 \pm 0.0011$
SGD,$66.64 \pm 2.70$,$0.8658 \pm 0.0518$,$43.4969$,$-0.8291 \pm 0.0028$
Synaptic Intelligence,$74.41 \pm 1.19$,$0.6586 \pm 0.1680$,$5.1065$,$-0.7239 \pm 0.0029$
STE,$29.35 \pm 0.96$,$0.6940 \pm 0.0425$,$9.3188$,$-0.5689 \pm 0.0105$
Synaptic Metaplasticity,$10.27 \pm 0.01$,$0.0006 \pm 0.0195$,$1.6446$,$-0.0009 \pm 0.0001$
BiMU,$90.30 \pm 0.38$,$0.9925 \pm 0.0029$,$139.4710$,$-0.8187 \pm 0.0014$


In [5]:
# make a table for footprint by converting to mb
footprint_table = pd.DataFrame()
for idx, row in df.iterrows():
    footprint = row["footprint"]/1e6
    opt = row["opt"]
    ewc = row["ewc"]
    si = row["si"]
    key = f"{opt}"
    if ewc != "none":
        key += f"-ewc"
    if si != "none":
        key += f"-si"
    if key == "adam":
        key = "STE"
    footprint_table[key] = [f"{footprint:.2f}MB"]
    
footprint_table = footprint_table.rename(index={0: "Memory Footprint"})
footprint_table = footprint_table.T
footprint_table.index = (
    footprint_table.index
    .str.replace("bayesbinn", "Bayes BiNN")
    .str.replace("bimu", "BiMU")
    .str.replace("mesu", "MESU")
    .str.replace("adamw", "Adam")
    .str.replace("adam", "STE")
    .str.replace("sgd-ewc", "EWC Online")
    .str.replace("sgd-si", "Synaptic Intelligence")
    .str.replace("sgd", "SGD")
    .str.replace("synapticmetaplasticity", "Synaptic Metaplasticity")
)
footprint_table

,Memory Footprint
Bayes BiNN,0.64MB
EWC Online,0.95MB
MESU,0.64MB
SGD,0.32MB
Synaptic Intelligence,0.95MB
STE,0.95MB
Synaptic Metaplasticity,1.84MB
BiMU,0.32MB
